<a href="https://colab.research.google.com/github/yoginikumar0608-gh/GenZSpace/blob/main/StudyMate_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-huggingface \
    chromadb \
    pypdf \
    sentence-transformers \
    streamlit \
    python-dotenv

In [6]:
!pip install -q langchain-chroma

In [1]:
import os
import shutil
from pathlib import Path

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from langchain_google_genai import ChatGoogleGenerativeAI

/tmp/ipykernel_1841/2761040365.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass(
    "Enter your Gemini API key: "
)

Enter your Gemini API key: ··········


In [3]:
DATA_DIR = "/content/documents"

os.makedirs(DATA_DIR, exist_ok=True)

print("Document folder created:")
print(DATA_DIR)

Document folder created:
/content/documents


In [4]:
uploaded = files.upload()

for filename, data in uploaded.items():
    filepath = os.path.join(DATA_DIR, filename)

    with open(filepath, "wb") as f:
        f.write(data)

print("\nUploaded files:")
for file in os.listdir(DATA_DIR):
    print("-", file)

Saving Module 5.pdf to Module 5.pdf

Uploaded files:
- Module 5.pdf


In [6]:
pdf_files = list(Path(DATA_DIR).glob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print("✓", pdf.name)

PDF files found: 1
✓ Module 5.pdf


In [7]:
documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()

    documents.extend(docs)

print("Total pages loaded:", len(documents))

Total pages loaded: 28


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 90


In [9]:
print("Example chunk:\n")
print(chunks[0].page_content[:1500])

print("\nMetadata:")
print(chunks[0].metadata)

Example chunk:

MODULE 5: TRENDS IN BIOENGINEERING (QUALITATIVE): 
Bioprinting techniques and materials, 3D printing of ear, bone and skin. 3D printed foods. Electrical 
tongue and electrical nose in food science, DNA origami and Biocomputing, Bioimaging and Artificial 
Intelligence for disease diagnosis. Self -healing Bioconcrete (based on bacillus spores, calcium lactate 
nutrients and biomineralization processes) and Bioremediation and Biomining via microbial surface 
adsorption (removal of heavy metals like Lead, Cadmium, Mercury, Arsenic). 
 
BIOENGINEERING 
Bioengineering is a discipline that applies  principles of biology and the tools of engineering to create 
usable, tangible, economically viable products. Examples of bioengineering research include  
 bacteria engineered to produce chemicals,  
 new medical imaging technology,  
 portable disease diagnostic devices, and  
 Tissue engineered organs.

Metadata:
{'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Wor

In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [11]:
CHROMA_PATH = "/content/chroma_db"

if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=CHROMA_PATH
)

print("Vector database created successfully.")

Vector database created successfully.


In [12]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

query = "What is machine learning?"

results = retriever.invoke(query)

print("Retrieved chunks:", len(results))

for i, doc in enumerate(results, 1):
    print("\n" + "=" * 60)
    print("RESULT", i)
    print("=" * 60)
    print(doc.page_content[:1000])
    print("\nSource:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))

Retrieved chunks: 4

RESULT 1
intervention and preventing complications. 
LIMITATIONS 
 Data Quality and Quantity: AI algorithms require large amounts of high -quality data to learn and 
make accurate predictions. In many healthcare settings, data may be incomplete, inconsistent, or 
biased, which can lead to inaccurate diagnoses. 
 Interpretability: Deep learning models, which are commonly used in AI for medical diagn osis, are 
often considered "black boxes" because they lack transparency and explainability.  They can be 
complex and difficult to understand, making it difficult for healthcare professionals to interpret the 
results.  
 Bias: AI algorithms may be biased leading to inaccurate or unfair diagnoses.  
 Limited Generalization: AI models may excel in diagnosing specific diseases they were trained on 
but may struggle with novel or rare diseases. They may not be able to generalize well to new or 
previously unseen conditions, which limit their utility.

Source: /content/

In [20]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

print("Gemini 3.6 Flash connected successfully.")

Gemini 3.6 Flash connected successfully.


In [15]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are StudyMate AI, an intelligent study assistant.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:

"I couldn't find the answer in the uploaded documents."

Do not invent information.

Explain difficult concepts clearly and in a student-friendly way.

Context:
{context}

Question:
{question}

Answer:
""")

In [16]:
def ask_rag(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)

    return response.content, retrieved_docs

In [23]:
question = "What is Bioengineering?"

answer, sources = ask_rag(question)

print("ANSWER")
print("=" * 60)
print(answer)

print("\n\nSOURCES")
print("=" * 60)

for doc in sources:
    print(
        f"- {Path(doc.metadata['source']).name}, "
        f"Page {doc.metadata.get('page', 'Unknown') + 1}"
    )

ANSWER
[{'type': 'text', 'text': '**Bioengineering** is a discipline that applies the principles of biology along with the tools of engineering to create usable, tangible, and economically viable products. \n\nExamples of bioengineering research include:\n* **Engineered bacteria:** Modifying bacteria so they can produce useful chemicals.\n* **Medical imaging technology:** Developing new ways to view the inside of the human body for medical diagnosis.\n* **Portable diagnostic devices:** Creating portable tools to quickly detect diseases.\n* **Tissue-engineered organs:** Building laboratory-grown tissues and organs for medical use.', 'extras': {'signature': 'Eu0QCuoQAWkUfROS6vm2x5ezf5eXJFk56LQQ9TypKAeBK+N7b53Mw7RCTdNXCVvyrwHh2gEaIm1GtgL2bQkiNj95tpCuFMPi1KJVd8C2otvL27aP1bbbkPJZjGKL/KOeXQFAfU8GuHwKbrue58YE7CbPeg9o/fMS7QLI0jvySRtT714kQ0QTdgteesXPyyJcCywNuiFsR8pB5vX/2KGGIvUqf6zpNJPTowf31yL7ucoZHgFGSA5KjoEag2u3gI8MvcxKFK4nXYA/jb3E7mcHbADI3cu1g+UNgjHDM4aJcDfanFsO+nJflc+lKL4oe4NwQzNxYV6LGSpM12e

In [25]:
while True:

    question = input("\nAsk StudyMate AI: ")

    if question.lower() in ["exit", "quit", "stop"]:
        print("StudyMate AI stopped.")
        break

    answer, sources = ask_rag(question)

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)

    print(answer)

    print("\n" + "=" * 70)
    print("SOURCES")
    print("=" * 70)

    unique_sources = set()

    for doc in sources:

        filename = Path(
            doc.metadata["source"]
        ).name

        page = doc.metadata.get("page", "Unknown") + 1

        source = f"{filename} — Page {page}"

        if source not in unique_sources:
            print("•", source)
            unique_sources.add(source)


Ask StudyMate AI: bioengineeering

ANSWER
[{'type': 'text', 'text': 'Based on the provided text, **bioengineering** is a field that applies the principles of biology along with engineering tools to design and create usable, tangible, and economically viable products. \n\nHere are some key examples of research in bioengineering mentioned in the document:\n* **Engineered Bacteria:** Modifying bacteria so they can produce useful chemicals.\n* **Medical Imaging Technology:** Developing new ways to visualize internal body structures (like bones and soft tissues).\n* **Portable Diagnostic Devices:** Creating easy-to-carry tools for diagnosing diseases.\n* **Tissue-Engineered Organs:** Creating replacement organs using engineering and biological techniques (such as 3D bioprinting).', 'extras': {'signature': 'Et4UCtsUAWkUfRPOKlzGcjiei0GJgC1XNRfVf3RftJ8lc2P9kMOriwE+Hq5mT55Ca+Fw6xst6dLQSEiOqn45s/3m9SURP9S8cgcyODOLkBIKW/364j9hp2+fW0N9ndGKCmVJa+NeLyiXd/pjSuREUd1hH6kqFiN3iDFnq3zPrkYvk5IQwSWipFOLre

KeyboardInterrupt: Interrupted by user

In [26]:
def study_action(action, topic):

    if action == "summary":
        question = f"""
        Give me a clear study summary of {topic}.
        Include the important concepts, definitions,
        and key points.
        """

    elif action == "mcq":
        question = f"""
        Create 5 multiple-choice questions about {topic}.
        Give four options for each question and identify
        the correct answer.
        """

    elif action == "exam":
        question = f"""
        Create a detailed 10-mark exam answer about {topic}.
        Structure it with introduction, explanation,
        important points, examples if available,
        and conclusion.
        """

    elif action == "simple":
        question = f"""
        Explain {topic} in very simple language.
        Use an example if the uploaded documents provide one.
        """

    else:
        question = topic

    answer, sources = ask_rag(question)

    return answer, sources

In [29]:
answer, sources = study_action(
    "summary",
    "bioengineering"
)

print(answer)

[{'type': 'text', 'text': 'Here is a clear, student-friendly study summary of **Bioengineering** based on your provided document:\n\n---\n\n# 📚 **Study Summary: Bioengineering**\n\n### **1. What is Bioengineering? (Definition)**\n**Bioengineering** is a discipline that combines the principles of biology with engineering tools to create usable, tangible, and economically viable products. \n\n**Examples of Bioengineering Research Include:**\n* Bacteria engineered to produce chemicals.\n* New medical imaging technologies.\n* Portable disease diagnostic devices.\n* Tissue-engineered organs.\n\n---\n\n### **2. Key Concepts & Trends in Bioengineering**\n\n#### **A. Biocomputing**\n* **Definition:** A field at the intersection of biology, engineering, and computer science that uses biological systems (cells, DNA, RNA, proteins) instead of electronic circuits to perform computational tasks like storing and processing data.\n* **DNA Computing:** Uses DNA strands to represent data and perform lo

In [31]:
answer, sources = study_action(
    "mcq",
    "bioengineering"
)

print(answer)

[{'type': 'text', 'text': 'Here are 5 multiple-choice questions based on the provided bioengineering text, along with student-friendly explanations for the answers:\n\n---\n\n### **Question 1**\n**What is the primary goal of bioengineering?**  \nA) To replace all biological research with electronic computer chips  \nB) To apply biology principles and engineering tools to create usable, tangible, and economically viable products  \nC) To study space physics using computer graphics  \nD) To eliminate all bacteria from industrial environments  \n\n* **Correct Answer:** **B) To apply biology principles and engineering tools to create usable, tangible, and economically viable products**  \n* **Explanation:** *Bioengineering is simply bringing biology and engineering together! It takes biological concepts and uses engineering tools to build real, practical, and affordable products like engineered bacteria or tissue-engineered organs.*\n\n---\n\n### **Question 2**\n**Which branch of biocomput

In [32]:
answer, sources = study_action(
    "exam",
    "machine learning"
)

print(answer)

[{'type': 'text', 'text': "I couldn't find the answer in the uploaded documents.", 'extras': {'signature': 'EpwVCpkVAWkUfRN6vXGv6nwhFdTaaX6tUQ3kSVEDD5oVJhoqQqinkGp6Fmdtcp7F8w3h2D5/rG3fhP30EuyvZ4UdLoGCEzSRi6LXKXiKwnTWYejm/1224lpHNNOVEdrwz75PYshhd4cR+MVPII8TAh+0vwluuIXR0lkFsB8WuQ0O16U9CIBHr0Jt2dKuZGBEtW3svJJGQ4x/sT9ZviZ8K7xht2azVe78bztck4pgQWQRtKhsZEKKuR4WtTQhpCEJFOrcrPolVhNpX5fCPmdWt0m09+03joUNAFs0NreaqlfqsioV4WVtYy2pUhL+9JDPfCR+WCZRp4bwq1+naY9VJGwoXYLU1t9rISxk9TrNTkfwonxlKHcCCcffTtQpyXfMLeoD6XgFfjh33WMfj0TWn8fSJrqVAyp1vWSFjXxmASnCZr+NZ0HxRiKGgEJ7+M8Gb9SQUIz6lDSCtEVXbc1gFZJPUPiqWwY3J262nTYa09YrhqBZ6GiCkovSlHM2fsinnEXupJPR/Vq8y5PPStM79xQ3o+nH9CEaXHuZ+kpwOiYDfKYV21ExAP88YB3Xgsjz1gVOb5dPa/1jIhERudTF5dkAgvZyhtODu8pofu0osKG8QoLwIByDTi729RB4W2URMcmRkYj49rD2iGI61YUL66/ybkzeuKfgczkVF+6MQrH5yR54A4+ljAXeLmyubBLfHQHrvQUn5HN8b3jOsYImizcdbaHZGT78cNFNxRuNoV2fU/OLHxsOFheuYO3NyvcwfjFdaNmc6UMbvDmgLwaXwKU/NeVWoEZo2Gml5y/LUNf7ACjJ9GQjhy1plyBt3zajFxx3PDy8r/JyyidnTed8aOmBAvtCEq8UbuIjOrP2+TxIZipuHD+IJl0pPDmm